### Transformation from Bronze to Silver
- This is data prep, clean-up and standardization saved to silver schema as a delta table
- Three tables: `rodent_complaints_silver`, `restaurant_violations_silver` (one row per violation), `restaurants_silver` (one row per restaurant).
- Every table and column gets a description, because Genie reads this metadata.

In [ ]:
%run ./00_config

#### `rodent_complaints_silver`
Confirmed rodent complaint = Rat Sighting + Signs of Rodents + Mouse Sighting. Condition Attracting Rodents is kept but flagged as not a sighting.

In [ ]:
%sql
CREATE OR REPLACE TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver AS
SELECT
  unique_key,
  regexp_extract(CAST(incident_zip AS STRING), '([0-9]{5})', 1)            AS zip,
  UPPER(TRIM(borough))                                                     AS borough,
  complaint_type,
  descriptor,
  descriptor IN ('Rat Sighting','Signs of Rodents','Mouse Sighting')       AS is_confirmed_sighting,
  CAST(created_date AS TIMESTAMP)                                          AS created_ts,
  CAST(closed_date  AS TIMESTAMP)                                          AS closed_ts,
  CASE WHEN closed_date IS NOT NULL
       THEN DATEDIFF(CAST(closed_date AS TIMESTAMP), CAST(created_date AS TIMESTAMP))
  END                                                                     AS days_to_close,
  status,
  status = 'Closed'                                                        AS is_closed,
  CAST(latitude  AS DOUBLE)                                                AS latitude,
  CAST(longitude AS DOUBLE)                                                AS longitude,
  -- lineage carried from bronze + silver stamp
  file_name,
  file_path,
  ingested_at,
  current_timestamp                                                        AS silver_loaded_at
FROM `nyc_rats`.`01_bronze`.rat_sightings_bronze
WHERE regexp_extract(CAST(incident_zip AS STRING), '([0-9]{5})', 1) <> '';

In [ ]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT zip) AS zips FROM `nyc_rats`.`02_silver`.rodent_complaints_silver;

#### `restaurant_violations_silver`  (grain: one row per violation)
Rows with a bad/empty ZIP are dropped here (they cannot join to a ZIP).

In [ ]:
%sql
CREATE OR REPLACE TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver AS
SELECT
  camis,
  dba,
  UPPER(TRIM(boro))                                                        AS boro,
  regexp_extract(CAST(zipcode AS STRING), '([0-9]{5})', 1)                 AS zip,
  cuisine_description,
  CAST(inspection_date AS TIMESTAMP)                                       AS inspection_ts,
  violation_code,
  violation_description,
  critical_flag,
  critical_flag = 'Critical'                                               AS is_critical,
  CAST(score AS INT)                                                       AS score,
  grade,
  violation_code IN ('04K','04L','08A')                                    AS is_rodent_violation,
  violation_code IN ('04K','04L','04M','04N','08A','08B')                  AS is_vermin_violation,
  -- lineage carried from bronze + silver stamp
  file_name,
  file_path,
  ingested_at,
  current_timestamp                                                        AS silver_loaded_at
FROM `nyc_rats`.`01_bronze`.restaurant_inspections_bronze
WHERE regexp_extract(CAST(zipcode AS STRING), '([0-9]{5})', 1) <> '';

In [ ]:
%sql
-- how many violation rows were dropped for a bad/empty ZIP
SELECT (SELECT COUNT(*) FROM `nyc_rats`.`01_bronze`.restaurant_inspections_bronze)
     - (SELECT COUNT(*) FROM `nyc_rats`.`02_silver`.restaurant_violations_silver) AS dropped_bad_zip_rows;

#### `restaurants_silver`  (grain: one row per restaurant)
Built from bronze so **all** distinct camis are kept (a restaurant whose ZIP is always blank still counts; it just gets a NULL zip and drops out of ZIP rollups). Row count should equal distinct camis.

In [ ]:
%sql
CREATE OR REPLACE TABLE `nyc_rats`.`02_silver`.restaurants_silver AS
WITH ranked AS (
  SELECT
    camis, dba, UPPER(TRIM(boro)) AS boro, cuisine_description, grade,
    CAST(score AS INT) AS score,
    NULLIF(regexp_extract(CAST(zipcode AS STRING), '([0-9]{5})', 1), '') AS zip,
    CAST(inspection_date AS TIMESTAMP) AS inspection_ts,
    violation_code, file_name, file_path, ingested_at,
    ROW_NUMBER() OVER (PARTITION BY camis
                       ORDER BY CAST(inspection_date AS TIMESTAMP) DESC NULLS LAST) AS rn
  FROM `nyc_rats`.`01_bronze`.restaurant_inspections_bronze
)
SELECT
  camis,
  ANY_VALUE(dba)                                       AS dba,
  MAX(zip)                                             AS zip,
  ANY_VALUE(boro)                                      AS boro,
  ANY_VALUE(cuisine_description)                       AS cuisine_description,
  MAX(CASE WHEN rn = 1 THEN grade END)                 AS latest_grade,
  MAX(CASE WHEN rn = 1 THEN score END)                 AS latest_score,
  MAX(inspection_ts)                                   AS last_inspection_ts,
  MAX(CASE WHEN violation_code IN ('04K','04L','08A') THEN 1 ELSE 0 END) = 1 AS ever_rodent,
  MAX(CASE WHEN violation_code IN ('04K','04L','04M','04N','08A','08B') THEN 1 ELSE 0 END) = 1 AS ever_vermin,
  -- lineage (aggregated across the restaurant's inspection rows)
  ANY_VALUE(file_name)                                 AS file_name,
  ANY_VALUE(file_path)                                 AS file_path,
  MAX(ingested_at)                                     AS ingested_at,
  current_timestamp                                    AS silver_loaded_at
FROM ranked
GROUP BY camis;

In [ ]:
%sql
SELECT COUNT(*) AS restaurants FROM `nyc_rats`.`02_silver`.restaurants_silver;   -- expect 26,114

#### Table & column descriptions (Genie reads these)

In [ ]:
%sql
COMMENT ON TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver IS
  'NYC 311 rodent complaints, cleaned. One row per 311 service request. Volume reflects who calls 311, not where rodents are.';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN unique_key COMMENT '311 service request id, one per complaint';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN zip COMMENT '5-digit ZIP the complaint was filed for';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN borough COMMENT 'NYC borough, upper-cased';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN descriptor COMMENT 'Rat Sighting, Signs of Rodents, Mouse Sighting, or Condition Attracting Rodents';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN is_confirmed_sighting COMMENT 'True for confirmed rodent presence (Rat/Signs/Mouse); False for Condition Attracting Rodents';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN created_ts COMMENT 'When the complaint was filed';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN closed_ts COMMENT 'When the complaint was closed, null if still open';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN days_to_close COMMENT 'Calendar days from created to closed, null if open';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN status COMMENT '311 status such as Closed or In Progress';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN is_closed COMMENT 'True when status is Closed';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN file_name COMMENT 'Lineage: source file the row was ingested from';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN file_path COMMENT 'Lineage: full volume path of the source file';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN ingested_at COMMENT 'Lineage: when the row was loaded into bronze';
ALTER TABLE `nyc_rats`.`02_silver`.rodent_complaints_silver ALTER COLUMN silver_loaded_at COMMENT 'Lineage: when the row was transformed into silver';

In [ ]:
%sql
COMMENT ON TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver IS
  'NYC restaurant health violations, cleaned. GRAIN: one row per violation, NOT per restaurant. Count restaurants with COUNT(DISTINCT camis).';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN camis COMMENT 'Restaurant id. Count distinct camis to count restaurants';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN dba COMMENT 'Restaurant name (doing business as)';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN zip COMMENT '5-digit ZIP of the restaurant';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN inspection_ts COMMENT 'Date of the inspection that produced this violation';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN violation_code COMMENT 'DOHMH violation code';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN violation_description COMMENT 'Human-readable violation text';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN is_critical COMMENT 'True when critical_flag is Critical';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN score COMMENT 'Inspection score, higher is worse';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN grade COMMENT 'Letter grade for the inspection, often blank on violation rows';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN is_rodent_violation COMMENT 'True for rodent-evidence codes 04K rats, 04L mice, 08A harborage';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN is_vermin_violation COMMENT 'True for the broader vermin set (adds roaches, flies, garbage)';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN file_name COMMENT 'Lineage: source file the row was ingested from';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN file_path COMMENT 'Lineage: full volume path of the source file';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN ingested_at COMMENT 'Lineage: when the row was loaded into bronze';
ALTER TABLE `nyc_rats`.`02_silver`.restaurant_violations_silver ALTER COLUMN silver_loaded_at COMMENT 'Lineage: when the row was transformed into silver';

In [ ]:
%sql
COMMENT ON TABLE `nyc_rats`.`02_silver`.restaurants_silver IS
  'One row per restaurant (distinct camis). Use this to count restaurants and get per-restaurant rodent evidence.';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN camis COMMENT 'Restaurant id, primary key of this table';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN dba COMMENT 'Restaurant name';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN zip COMMENT '5-digit ZIP, null if the restaurant has no usable ZIP';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN cuisine_description COMMENT 'Cuisine category';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN latest_grade COMMENT 'Grade from the most recent inspection';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN latest_score COMMENT 'Score from the most recent inspection';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN last_inspection_ts COMMENT 'Date of the most recent inspection';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN ever_rodent COMMENT 'True if ever cited for rodent evidence (04K, 04L, 08A)';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN ever_vermin COMMENT 'True if ever cited for any broader vermin violation';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN file_name COMMENT 'Lineage: source file the restaurant was ingested from';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN file_path COMMENT 'Lineage: full volume path of the source file';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN ingested_at COMMENT 'Lineage: latest bronze ingest time across the restaurant inspection rows';
ALTER TABLE `nyc_rats`.`02_silver`.restaurants_silver ALTER COLUMN silver_loaded_at COMMENT 'Lineage: when the restaurant row was built in silver';

#### Verify

In [ ]:
%sql
SHOW TABLES IN `nyc_rats`.`02_silver`;